# Fine-tuned request classifier: live demo
Llama 3.2 3B + QLoRA (GGUF Q4_K_M), downloaded from Hugging Face and run here with llama.cpp.
The model only classifies. Routing is plain code, shown in the output. **Built with Llama.**

Model: https://huggingface.co/Sanved13/Llama-3.2-3B-request-classifier

Run the cells top to bottom. A CPU runtime is enough; each answer takes roughly 15 to 60 seconds.
The first cell installs llama.cpp and may take a few minutes.

In [ ]:
%%capture
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
!pip install -U huggingface_hub

In [ ]:
from llama_cpp import Llama
llm = Llama.from_pretrained(
    repo_id="Sanved13/Llama-3.2-3B-request-classifier",
    filename="req-classifier.Q4_K_M.gguf",
    n_ctx=2048, verbose=False)
print("model loaded")

In [ ]:
import json, re

THRESHOLD = 0.70
SYSTEM = ("You are a customer-request classifier. Read the request and reply with ONLY a JSON object "
          "with keys: request_type, urgency, sub_topic, confidence, summary, customer_sentiment, "
          "entities (invoice_number, amount, currency), required_action.")

def classify(subject, message):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Subject: {subject.strip()}\nMessage: {message.strip()}"}]
    raw = llm.create_chat_completion(messages=msgs, temperature=0, max_tokens=320)["choices"][0]["message"]["content"]
    try:
        obj = json.loads(re.search(r"\{.*\}", raw, re.S).group(0))
        conf = float(obj["confidence"])
    except Exception:
        return "INVALID OUTPUT -> human review", {"raw_output": raw}
    # deterministic routing, exactly as in the n8n workflow
    if obj.get("request_type") == "urgent_escalation" or obj.get("urgency") == "critical":
        route = "URGENT ESCALATION: HUMAN REVIEW REQUIRED"
    elif conf < THRESHOLD:
        route = f"LOW CONFIDENCE ({conf:.2f} < {THRESHOLD}) -> human review"
    else:
        route = f"Auto-route: {obj.get('request_type')} (confidence {conf:.2f})"
    return route, obj

## Try the five test cases from the project

In [ ]:
tests = [
    ("Wrong invoice amount", "I was charged $450 on invoice INV-20931 but my plan is $320 a month. Please correct this and refund the difference."),
    ("Support hours", "Hi, what are your customer support hours on weekends?"),
    ("Account security issue", "Someone logged into my account from another country and changed my password. I'm locked out and I see transfers I didn't make. Please help right now."),
    ("Need invoice copy", "URGENT!!! Please send me a copy of invoice INV-55120 today."),
    ("Help", "I need help with something."),
]
for s, m in tests:
    route, obj = classify(s, m)
    print(f"[{s}]  ->  {route}")
    print(json.dumps(obj, indent=1)[:600], "\n")

## Or try your own (creates a temporary public link, valid for a few days)

In [ ]:
import gradio as gr
demo = gr.Interface(
    fn=classify,
    inputs=[gr.Textbox(label="Subject"), gr.Textbox(label="Message", lines=5)],
    outputs=[gr.Textbox(label="Routing decision"), gr.JSON(label="Model output")],
    title="Fine-tuned customer-request classifier")
demo.launch(share=True)